# 03 · Majority-class baseline

The floor every model must beat: always predict the majority class of `KDDTrain+`. The predicted class is learned from the training labels, not hard-coded. Metrics use the same function as every other model (`cf.compute_comprehensive_metrics`). Its score is constant, so its ROC-AUC is 0.5 by construction.

**Outputs:** `models/majority_baseline.joblib`, `results/metrics/majority_baseline.json`, `results/predictions/majority_baseline_predictions.csv`.

In [1]:
import sys, json, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from src.config import set_all_seeds
from src import controlled_framework as cf

rng = set_all_seeds(cf.SEED)
sns.set_theme(style="whitegrid", context="notebook")
print(f"primary seed={cf.SEED} | seeds={cf.SEEDS} | trials={cf.N_TRIALS} | novelty CV folds={cf.N_SPLITS} | "
      f"threshold={cf.THRESHOLD} | FPR budgets={cf.FPR_BUDGETS} | class_weight={cf.CLASS_WEIGHT}")

primary seed=42 | seeds=[42, 1, 2, 3, 4] | trials=25 | novelty CV folds=5 | threshold=0.5 | FPR budgets=[0.01, 0.02, 0.05] | class_weight=None


In [2]:
import time, joblib
from sklearn.dummy import DummyClassifier

data = cf.load_dataset()
t0 = time.perf_counter(); clf = DummyClassifier(strategy="most_frequent").fit(data.X_train, data.y_train); fit_s = time.perf_counter() - t0
t0 = time.perf_counter(); y_prob = clf.predict_proba(data.X_test)[:, 1]; infer_s = time.perf_counter() - t0
y_pred = (y_prob > cf.THRESHOLD).astype(int)
majority = int(clf.classes_[np.argmax(clf.class_prior_)])
print(f"training majority class = {majority} ({'normal' if majority == 0 else 'attack'}), prior = {clf.class_prior_.round(4)}")

training majority class = 0 (normal), prior = [0.5346 0.4654]


In [3]:
metrics, preds = cf.compute_comprehensive_metrics(
    data.y_test.to_numpy(), y_pred, data.test_df, data.unseen_attacks, "Majority Baseline",
    {"strategy": "most_frequent", "predicted_class": majority}, fit_s, infer_s, y_prob=y_prob)
metrics["seed"] = cf.SEED

cf.ensure_dirs(cf.MODELS_DIR, cf.METRICS_DIR, cf.PRED_DIR)
joblib.dump(clf, cf.MODELS_DIR / "majority_baseline.joblib")
(cf.METRICS_DIR / "majority_baseline.json").write_text(json.dumps(metrics, indent=2))
preds.to_csv(cf.PRED_DIR / "majority_baseline_predictions.csv", index=False)
pd.Series(cf.summary_row(metrics)).to_frame("Majority Baseline")

,Majority Baseline
model,Majority Baseline
accuracy,0.430732
macro_f1,0.301057
attack_precision,0.0
attack_recall,0.0
false_positive_rate,0.0
seen_recall,0.0
unseen_recall,0.0
generalization_drop_pp,0.0
reweighted_drop_pp,0.0


**Reading.** Predicting *normal* for every row gives accuracy equal to the test share of normal traffic, zero attack recall, a Macro-F1 of about 0.30 and ROC-AUC 0.5. Accuracy is therefore not used to rank models.